# Modelling SBERT

Ekspor model `intfloat/multilingual-e5-small` dengan Mean Pooling dan L2 Normalization ke ONNX dan kuantisasi.

*Catatan: Model E5 bekerja secara asimetris. Untuk performa pencocokan terbaik, tambahkan prefiks `query: ` pada kueri pencarian (lowongan/JD) dan `passage: ` pada dokumen pencarian (CV) sebelum tokenisasi.*

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
import numpy as np

In [2]:
# Patch untuk menghindari WinError 32 PermissionError di Windows saat kuantisasi
import sys
import onnxruntime.quantization.quant_utils as quant_utils

def patched_load_model_with_shape_infer(model_path):
    import onnx
    from pathlib import Path
    inferred_model_path = quant_utils.generate_identified_filename(Path(model_path), '-inferred')
    onnx.shape_inference.infer_shapes_path(str(model_path), str(inferred_model_path))
    model = onnx.load(inferred_model_path.as_posix())
    quant_utils.add_infer_metadata(model)
    try:
        inferred_model_path.unlink()
    except PermissionError:
        pass # Abaikan error permission di Windows
    return model

quant_utils.load_model_with_shape_infer = patched_load_model_with_shape_infer
if 'onnxruntime.quantization.quantize' in sys.modules:
    sys.modules['onnxruntime.quantization.quantize'].load_model_with_shape_infer = patched_load_model_with_shape_infer

In [3]:
model_id = 'intfloat/multilingual-e5-small'

tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModel.from_pretrained(model_id)

In [4]:
tokenizer.save_pretrained('./sbert_tokenizer')

('./sbert_tokenizer\\tokenizer_config.json',
 './sbert_tokenizer\\special_tokens_map.json',
 './sbert_tokenizer\\tokenizer.json')

In [5]:
class SBERTPooler(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
        
    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        token_embeddings = outputs.last_hidden_state
        
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        embeddings = sum_embeddings / sum_mask
        
        # L2 normalization
        embeddings = F.normalize(embeddings, p=2, dim=1)
        return embeddings

model = SBERTPooler(base_model)
model.eval()

SBERTPooler(
  (base_model): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(250037, 384, padding_idx=0)
      (position_embeddings): Embedding(512, 384)
      (token_type_embeddings): Embedding(2, 384)
      (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
              (LayerNorm): LayerNorm((384,), eps=1e-12, elemen

In [6]:
dummy_text = 'John Doe lives in New York and works as an Engineer.'
dummy_input = tokenizer(dummy_text, return_tensors='pt')
dummy_input

{'input_ids': tensor([[    0,  4939,   984,    13, 60742,    23,  2356,  5753,   136, 43240,
           237,   142, 90125,    56,     5,     2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [7]:
onnx_model_path = 'onnx_SBERT/sbert_embedding.onnx'

torch.onnx.export(
    model,
    (dummy_input['input_ids'], dummy_input['attention_mask']),
    onnx_model_path,
    input_names=['input_ids', 'attention_mask'],
    output_names=['embeddings'],
    dynamic_axes={'input_ids': {0: 'batch_size', 1: 'sequence_length'},
                  'attention_mask': {0: 'batch_size', 1: 'sequence_length'},
                  'embeddings': {0: 'batch_size'}},
    opset_version=14,
    do_constant_folding=True
)
print(f'Model exported to {onnx_model_path}')

d:\Hasil_Coding\Capstone_Project\model\.venv\lib\site-packages\transformers\modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


Model exported to onnx_SBERT/sbert_embedding.onnx


In [8]:
quantized_model_path = 'onnx_SBERT/sbert_embedding_quantized.onnx'

quantize_dynamic(
    onnx_model_path,
    quantized_model_path,
    weight_type=QuantType.QUInt8
)
print(f'Quantized model exported to {quantized_model_path}')

Quantized model exported to onnx_SBERT/sbert_embedding_quantized.onnx


In [9]:
session = ort.InferenceSession(quantized_model_path)
inputs = {
    'input_ids': dummy_input['input_ids'].numpy(),
    'attention_mask': dummy_input['attention_mask'].numpy()
}
outputs = session.run(None, inputs)

print('Embeddings shape:', outputs[0].shape)
print('Sample embeddings:', outputs[0][0][:5])

Embeddings shape: (1, 384)
Sample embeddings: [ 0.02816573  0.00741282 -0.08209322 -0.01429461  0.09988953]
